In [8]:
import os
from dss import dss

# 1. Path to your Master_withPV.dss file
dss_file = r"C:/Program Files/OpenDSS/EPRITestCircuits/epri_dpv/J1/Master_withPV.dss"

# 2. Compile circuit (Text command interface)
dss.Text.Command = f'redirect "{dss_file}"'

# 3. Solve power flow (Solution lives under ActiveCircuit)
dss.ActiveCircuit.Solution.Solve()

# 4. Verify network state & PV systems
if dss.ActiveCircuit.Solution.Converged:
    print("✅ EPRI J1 Feeder (With PV) loaded and solved successfully!\n")
    print(f"Total Buses: {len(dss.ActiveCircuit.AllBusNames)}")
    
    # Note the capital 'S' in PVSystems
    pv_names = dss.ActiveCircuit.PVSystems.AllNames
    
    print(f"Total Pre-configured PV Systems: {len(pv_names)}")
    if pv_names:
        print(f"Sample PV System Names: {pv_names[:5]}")
else:
    print("❌ Power flow did not converge. Check file paths.")

✅ EPRI J1 Feeder (With PV) loaded and solved successfully!

Total Buses: 3434
Total Pre-configured PV Systems: 13
Sample PV System Names: ['3p_existingsite1', '3p_existingsite2', '3p_existingsite3', '3p_existingsite4', 'c_existing2']


In [13]:
# Cell 2: Voltage Profile & Constraint Analysis

# 1. Voltage Profile 
active_voltages = [v for v in dss.ActiveCircuit.AllBusVmagPu if v > 0.1]

if len(active_voltages) > 0:
    print("=== VOLTAGE PROFILE ===")
    print(f"Min Voltage: {min(active_voltages):.4f} p.u.")
    print(f"Max Voltage: {max(active_voltages):.4f} p.u.")
    print(f"Undervoltage Nodes (< 0.95 p.u.): {sum(1 for v in active_voltages if v < 0.95)}")
    print(f"Overvoltage Nodes (> 1.05 p.u.): {sum(1 for v in active_voltages if v > 1.05)}")

# 2. Line Loading & Constraint Check
overloaded_lines = []

# Initialize loop: First returns the index (should be > 0)
idx = dss.ActiveCircuit.Lines.First  

while idx > 0:
    currents = dss.ActiveCircuit.ActiveCktElement.CurrentsMagAng
    
    max_I = max(currents[0::2]) if len(currents) > 0 else 0
    rated_I = dss.ActiveCircuit.Lines.NormAmps
    
    if rated_I > 0 and (max_I / rated_I) > 1.0:
        pct = (max_I / rated_I) * 100
        overloaded_lines.append((dss.ActiveCircuit.Lines.Name, round(pct, 1)))
        
    # Advances to the next line. Returns 0 when finished.
    idx = dss.ActiveCircuit.Lines.Next  

print("\n=== CONSTRAINT ANALYSIS ===")
print(f"Total Overloaded Lines (>100% capacity): {len(overloaded_lines)}")
if overloaded_lines:
    print(f"Sample Overloads: {overloaded_lines[:3]}")

=== VOLTAGE PROFILE ===
Min Voltage: 0.9679 p.u.
Max Voltage: 1.0414 p.u.
Undervoltage Nodes (< 0.95 p.u.): 0
Overvoltage Nodes (> 1.05 p.u.): 0

=== CONSTRAINT ANALYSIS ===
Total Overloaded Lines (>100% capacity): 0


In [15]:
import os
import dss
import pandas as pd

# Define the exact path
file_path = r"C:\Users\Smarterise PC\Projects\PoC\data\epri_j1\Master_withPV.dss"

# 1. SAFETY CHECK: Does Python actually see the file?
if not os.path.exists(file_path):
    print(f"🛑 STOP: Python cannot find the file here:\n{file_path}")
    print("Please check your folder. Is the file inside a subfolder (like data\\epri_j1\\J1\\...)?")
else:
    print("✅ File found! Booting up OpenDSS...")
    
    # Initialize OpenDSS engine
    dss_engine = dss.DSS
    dss_text = dss_engine.Text
    dss_circuit = dss_engine.ActiveCircuit

    # Clear any old data out of the engine's memory
    dss_text.Command = "clear"

    # 2. THE FIX: Using double quotes for the compile command so OpenDSS reads the space in "Smarterise PC"
    dss_text.Command = f'compile "{file_path}"'

    # Solve the steady-state power flow
    dss_text.Command = "solve"

    print(f"🎉 Network Solved Successfully!")
    print(f"Total Buses: {dss_circuit.NumBuses}")
    print(f"Total Nodes: {dss_circuit.NumNodes}")

🛑 STOP: Python cannot find the file here:
C:\Users\Smarterise PC\Projects\PoC\data\epri_j1\Master_withPV.dss
Please check your folder. Is the file inside a subfolder (like data\epri_j1\J1\...)?


In [16]:
import os
import dss
import pandas as pd

# Define the exact path you provided
file_path = r"C:\Program Files\OpenDSS\EPRITestCircuits\epri_dpv\J1\Master_withPV.dss"

# 1. SAFETY CHECK: Does Python actually see the file?
if not os.path.exists(file_path):
    print(f"🛑 STOP: Python cannot find the file here:\n{file_path}")
else:
    print("✅ File found! Booting up OpenDSS...")
    
    # Initialize OpenDSS engine
    dss_engine = dss.DSS
    dss_text = dss_engine.Text
    dss_circuit = dss_engine.ActiveCircuit

    # Clear any old data out of the engine's memory
    dss_text.Command = "clear"

    # 2. Compile using double quotes
    dss_text.Command = f'compile "{file_path}"'

    # Solve the steady-state power flow
    dss_text.Command = "solve"

    print(f"🎉 Network Solved Successfully!")
    print(f"Total Buses: {dss_circuit.NumBuses}")
    print(f"Total Nodes: {dss_circuit.NumNodes}")

✅ File found! Booting up OpenDSS...
🎉 Network Solved Successfully!
Total Buses: 3434
Total Nodes: 4245


In [19]:
# Create lists to hold our extracted data
bus_names = []
base_kvs = []
pu_voltages = []

print("Extracting actual voltage magnitudes...")

# Loop through every bus in the simulated network
for bus_name in dss_circuit.AllBusNames:
    dss_circuit.SetActiveBus(bus_name)
    bus = dss_circuit.ActiveBus
    
    # USE puVmagAngle: Returns [Magnitude, Angle, Magnitude, Angle...]
    mag_angles = bus.puVmagAngle
    
    if len(mag_angles) > 0:
        bus_names.append(bus_name)
        base_kvs.append(bus.kVBase)
        # Index 0 is the Magnitude of Phase 1, Index 1 is the Angle
        pu_voltages.append(mag_angles[0])

# Convert to Pandas DataFrame
df_buses = pd.DataFrame({
    'Bus_ID': bus_names,
    'Base_kV': base_kvs,
    'Voltage_PU': pu_voltages
})

# Filter for real Voltage Violations (Under 0.95 p.u. but above 0.0 to ignore dead lines)
df_violations = df_buses[(df_buses['Voltage_PU'] < 0.95) & (df_buses['Voltage_PU'] > 0.1)].copy()

# Sort to find the worst voltage drops
df_violations = df_violations.sort_values(by='Voltage_PU')

print(f"Extraction complete! Found {len(df_violations)} valid under-voltage buses.")
display(df_violations.head(5))

# ==========================================
# THE BRIDGE TO YOUR HTML DASHBOARD
# ==========================================
# Export the top 15 worst constraints to a JSON file
export_path = r"C:\Users\Smarterise PC\Projects\PoC\data\epri_j1\constraint_register.json"
df_violations.head(15).to_json(export_path, orient="records", indent=4)

print(f"\n✅ Top 15 constraints exported successfully to:\n{export_path}")

Extracting actual voltage magnitudes...
Extraction complete! Found 0 valid under-voltage buses.


,Bus_ID,Base_kV,Voltage_PU



✅ Top 15 constraints exported successfully to:
C:\Users\Smarterise PC\Projects\PoC\data\epri_j1\constraint_register.json
